In [6]:
import json
from huggingface_hub import HfApi, hf_hub_download
from dotenv import load_dotenv
load_dotenv()

api = HfApi()

models = api.list_models(filter="text-generation", sort="downloads", limit=10)

print(f"{'Model ID':<50} | {'Max Token Limit':<15}")
print("-" * 68)

for m in models:
  try:
    # Download only the small config.json file for the model
    config_path = hf_hub_download(
        repo_id=m.id, filename="config.json", repo_type="model"
    )

    with open(config_path, "r") as f:
      config = json.load(f)

    # Check common keys used for token/context limits
    max_tokens = (
        config.get("max_position_embeddings")
        or config.get("n_ctx")
        or config.get("model_max_length")
        or config.get("max_seq_len")
        or "Not Specified"
    )

    print(f"{m.id:<50} | {max_tokens:<15}")

  except Exception:
    # Handles cases where config.json doesn't exist or requires access tokens
    print(f"{m.id:<50} | Error Loading")

Model ID                                           | Max Token Limit
--------------------------------------------------------------------
Qwen/Qwen3-0.6B                                    | 40960          
trl-internal-testing/tiny-Qwen2ForCausalLM-2.5     | 32768          
openai-community/gpt2                              | 1024           
Qwen/Qwen3-8B                                      | 40960          
unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF          | Error Loading
nvidia/Qwen3.6-35B-A3B-NVFP4                       | Not Specified  
Qwen/Qwen2.5-7B-Instruct                           | 32768          
facebook/opt-125m                                  | 2048           
Qwen/Qwen2.5-3B-Instruct                           | 32768          
Qwen/Qwen2.5-1.5B-Instruct                         | 32768          


In [15]:
"""
Script to fetch text-generation models from Hugging Face Hub
and retrieve their token limits (context length) from config.json.

Reads HF_TOKEN from a .env file in the same directory.
"""

import os
import time
import requests
from huggingface_hub import HfApi
from tqdm import tqdm
from typing import Optional, List, Tuple, Dict, Any
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv   # <-- pip install python-dotenv

# ---------- Load environment variables from .env ----------
load_dotenv()  # Looks for .env in current directory
HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    print("⚠️  WARNING: HF_TOKEN not found in .env file.")
    print("   You may experience rate-limiting. Please set HF_TOKEN in .env")
    print("   Example: HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxx")
    print()

# ---------- CONFIG ----------
MODEL_LIMIT = 200          # number of models to fetch (None for all)
SORT_BY = "downloads"      # "downloads", "likes", or "trending"
DISPLAY_TOP = 50
REQUEST_DELAY = 0.3        # seconds between models to avoid rate limits
RETRIES = 2
# -----------------------------

# Create a session with retries
session = requests.Session()
retry_strategy = Retry(
    total=RETRIES,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

if HF_TOKEN:
    session.headers.update({"Authorization": f"Bearer {HF_TOKEN}"})

api = HfApi(token=HF_TOKEN)  # pass token even if None


def find_max_position_embeddings(config: Dict[str, Any]) -> Optional[int]:
    """Recursively search for a token-limit key in the config dict."""
    keys = [
        "max_position_embeddings",
        "max_sequence_length",
        "n_positions",
        "max_length",
        "model_max_length",
        "max_len"
    ]
    # Check top-level
    for key in keys:
        if key in config and isinstance(config[key], int):
            return config[key]

    # Check nested structures (e.g., "text_config", "config")
    for nested_key in ["text_config", "config", "model_config"]:
        if nested_key in config and isinstance(config[nested_key], dict):
            nested = config[nested_key]
            for key in keys:
                if key in nested and isinstance(nested[key], int):
                    return nested[key]

    return None


def fetch_json_from_repo(model_id: str, filename: str) -> Optional[Dict[str, Any]]:
    """Fetch a JSON file from the model's repo, with retries."""
    url = f"https://huggingface.co/{model_id}/raw/main/{filename}"
    try:
        resp = session.get(url, timeout=15)
        if resp.status_code == 200:
            return resp.json()
    except Exception:
        pass
    return None


def get_model_token_limit(model_id: str) -> Optional[int]:
    """Try config.json then generation_config.json."""
    # Primary: config.json
    config = fetch_json_from_repo(model_id, "config.json")
    if config:
        limit = find_max_position_embeddings(config)
        if limit is not None:
            return limit

    # Secondary: generation_config.json
    gen = fetch_json_from_repo(model_id, "generation_config.json")
    if gen:
        for key in ["max_length", "max_position_embeddings"]:
            if key in gen and isinstance(gen[key], int):
                return gen[key]

    return None


def get_text_generation_models(
    limit: Optional[int] = None,
    sort_by: str = "downloads"
) -> List[Tuple[str, Optional[int]]]:
    print(f"Fetching text-generation models (sort: {sort_by})...")
    models = list(api.list_models(
        filter="text-generation",
        sort=sort_by,
        limit=limit
    ))
    print(f"Found {len(models)} models. Fetching token limits...")

    results = []
    for model in tqdm(models, desc="Fetching token limits"):
        token_limit = get_model_token_limit(model.id)
        results.append((model.id, token_limit))
        time.sleep(REQUEST_DELAY)

    return results


def print_results(results: List[Tuple[str, Optional[int]]], top_n: int = 50):
    """Print a sorted table of models and token limits."""
    # Sort by token limit (descending), None at the end
    sorted_results = sorted(
        results,
        key=lambda x: (x[1] is None, x[1] if x[1] is not None else 0),
        reverse=True
    )

    print("\n" + "=" * 80)
    print(f"{'Model ID':<50} {'Token Limit':<20}")
    print("=" * 80)

    count = 0
    for model_id, token_limit in sorted_results:
        if count >= top_n:
            break
        token_str = f"{token_limit:,}" if token_limit is not None else "Unknown"
        print(f"{model_id:<50} {token_str:<20}")
        count += 1

    known = [limit for _, limit in results if limit is not None]
    print("\n" + "=" * 80)
    print(f"Total models fetched: {len(results)}")
    print(f"Models with known token limits: {len(known)}")
    if known:
        print(f"Max token limit: {max(known):,}")
        print(f"Min token limit: {min(known):,}")
        print(f"Average token limit: {sum(known) // len(known):,}")


def main():
    print("Hugging Face Text-Generation Models & Token Limits")
    print("=" * 80)
    print(f"Authentication: {'✅ Yes (from .env)' if HF_TOKEN else '❌ No (rate-limited)'}")
    print("Token limits are extracted from config.json / generation_config.json.")
    print("Models that are GGUF or lack config will show Unknown.")
    print("=" * 80)

    try:
        results = get_text_generation_models(limit=MODEL_LIMIT, sort_by=SORT_BY)
        print_results(results, top_n=DISPLAY_TOP)
    except Exception as e:
        print(f"An error occurred: {e}")


if __name__ == "__main__":
    main()

Hugging Face Text-Generation Models & Token Limits
Authentication: ✅ Yes (from .env)
Token limits are extracted from config.json / generation_config.json.
Models that are GGUF or lack config will show Unknown.
Fetching text-generation models (sort: downloads)...
Found 200 models. Fetching token limits...


Fetching token limits: 100%|████████████████████████████████████████████████████████| 200/200 [02:01<00:00,  1.64it/s]


Model ID                                           Token Limit         
unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF          Unknown             
meta-llama/Llama-3.2-1B-Instruct                   Unknown             
meta-llama/Llama-3.1-8B-Instruct                   Unknown             
ornith-ai/Ornith-1.0-9B-GGUF                       Unknown             
google/gemma-3-1b-it                               Unknown             
ornith-ai/Ornith-1.0-35B-GGUF                      Unknown             
ornith-ai/Ornith-1.5-9B-GGUF                       Unknown             
ornith-ai/Ornith-1.5-35B-A3B-GGUF                  Unknown             
JonathanColetti/Qwen3.8-27B-Uncensored-GGUF        Unknown             
antirez/deepseek-v4-gguf                           Unknown             
vikhyatk/moondream2                                Unknown             
meta-llama/Meta-Llama-3-8B-Instruct                Unknown             
google/gemma-3-270m                                Unknown     

In [16]:
"""
Fetch text-generation models from Hugging Face Hub and retrieve their token limits.
Reads HF_TOKEN from .env. Distinguishes between Unknown (no config) and Gated (access denied).
"""

import os
import time
import requests
from huggingface_hub import HfApi
from tqdm import tqdm
from typing import Optional, List, Tuple, Dict, Any
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv

# ---------- Load .env ----------
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    print("⚠️  WARNING: HF_TOKEN not set in .env. You may be rate-limited.")

# ---------- Configuration ----------
MODEL_LIMIT = 200
SORT_BY = "downloads"
DISPLAY_TOP = 50
REQUEST_DELAY = 0.3
RETRIES = 2

# ---------- Session with retries ----------
session = requests.Session()
retry_strategy = Retry(
    total=RETRIES,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)
if HF_TOKEN:
    session.headers.update({"Authorization": f"Bearer {HF_TOKEN}"})

api = HfApi(token=HF_TOKEN)

# Try to use the official get_model_config (available in huggingface_hub >= 0.19.0)
try:
    from huggingface_hub import get_model_config
    USE_OFFICIAL = True
except ImportError:
    USE_OFFICIAL = False
    print("⚠️  huggingface_hub.get_model_config not found. Falling back to requests.")


def find_max_position_embeddings(config: Dict[str, Any]) -> Optional[int]:
    """Recursively search for token-limit keys."""
    keys = [
        "max_position_embeddings",
        "max_sequence_length",
        "n_positions",
        "max_length",
        "model_max_length",
        "max_len"
    ]
    # Top-level
    for key in keys:
        if key in config and isinstance(config[key], int):
            return config[key]
    # Nested
    for nested_key in ["text_config", "config", "model_config"]:
        if nested_key in config and isinstance(config[nested_key], dict):
            nested = config[nested_key]
            for key in keys:
                if key in nested and isinstance(nested[key], int):
                    return nested[key]
    return None


def fetch_json_from_repo(model_id: str, filename: str) -> Tuple[Optional[Dict], Optional[str]]:
    """
    Fetch JSON from repo. Returns (data, status) where status is None on success,
    or a string like "403", "404", "error".
    """
    url = f"https://huggingface.co/{model_id}/raw/main/{filename}"
    try:
        resp = session.get(url, timeout=15)
        if resp.status_code == 200:
            return resp.json(), None
        else:
            return None, str(resp.status_code)
    except Exception:
        return None, "error"


def get_model_token_limit(model_id: str) -> Tuple[Optional[int], Optional[str]]:
    """
    Returns (token_limit, status) where status is None if found,
    "Gated" for 403, "NotFound" for 404, "Error" for others.
    """
    # Official method if available
    if USE_OFFICIAL:
        try:
            config = get_model_config(model_id, timeout=15)
            limit = find_max_position_embeddings(config)
            if limit is not None:
                return limit, None
            # Try generation_config as fallback via requests
        except Exception as e:
            # If the error is about gated access
            if "403" in str(e) or "gated" in str(e).lower():
                return None, "Gated"
            else:
                return None, "Error"

    # Fallback: requests
    config, status = fetch_json_from_repo(model_id, "config.json")
    if config is not None:
        limit = find_max_position_embeddings(config)
        if limit is not None:
            return limit, None
    elif status == "403":
        return None, "Gated"
    elif status == "404":
        return None, "NotFound"
    elif status is not None:
        return None, "Error"

    # Try generation_config.json
    gen, gen_status = fetch_json_from_repo(model_id, "generation_config.json")
    if gen is not None:
        for key in ["max_length", "max_position_embeddings"]:
            if key in gen and isinstance(gen[key], int):
                return gen[key], None
    elif gen_status == "403":
        return None, "Gated"

    return None, "NotFound"


def get_text_generation_models(
    limit: Optional[int] = None,
    sort_by: str = "downloads"
) -> List[Tuple[str, Optional[int], Optional[str]]]:
    """Return list of (model_id, token_limit, status)."""
    print(f"Fetching text-generation models (sort: {sort_by})...")
    models = list(api.list_models(
        filter="text-generation",
        sort=sort_by,
        limit=limit
    ))
    print(f"Found {len(models)} models. Fetching token limits...")

    results = []
    for model in tqdm(models, desc="Fetching token limits"):
        limit, status = get_model_token_limit(model.id)
        results.append((model.id, limit, status))
        time.sleep(REQUEST_DELAY)
    return results


def print_results(results: List[Tuple[str, Optional[int], Optional[str]]], top_n: int = 50):
    # Sort: known limits first, then Unknown, then Gated, etc.
    def sort_key(item):
        limit, status = item[1], item[2]
        if limit is not None:
            return (0, -limit)  # descending limit
        elif status == "Gated":
            return (2, 0)
        else:
            return (1, 0)  # Unknown

    sorted_results = sorted(results, key=sort_key)

    print("\n" + "=" * 80)
    print(f"{'Model ID':<50} {'Token Limit':<15} {'Status':<10}")
    print("=" * 80)

    count = 0
    for model_id, limit, status in sorted_results:
        if count >= top_n:
            break
        if limit is not None:
            token_str = f"{limit:,}"
            status_str = "OK"
        else:
            token_str = "Unknown"
            status_str = status if status else "NotFound"
        print(f"{model_id:<50} {token_str:<15} {status_str:<10}")
        count += 1

    known = [lim for _, lim, _ in results if lim is not None]
    gated = [m for m, _, st in results if st == "Gated"]
    print("\n" + "=" * 80)
    print(f"Total models fetched: {len(results)}")
    print(f"Models with known token limits: {len(known)}")
    print(f"Models that are Gated (license required): {len(gated)}")
    if known:
        print(f"Max token limit: {max(known):,}")
        print(f"Min token limit: {min(known):,}")
        print(f"Average token limit: {sum(known) // len(known):,}")


def main():
    print("Hugging Face Text-Generation Models & Token Limits")
    print("=" * 80)
    print(f"Authentication: {'✅ Yes (from .env)' if HF_TOKEN else '❌ No'}")
    print("Status: OK = limit found, Gated = license required, NotFound = no config")
    print("=" * 80)

    try:
        results = get_text_generation_models(limit=MODEL_LIMIT, sort_by=SORT_BY)
        print_results(results, top_n=DISPLAY_TOP)
    except Exception as e:
        print(f"An error occurred: {e}")


if __name__ == "__main__":
    main()

⚠️  huggingface_hub.get_model_config not found. Falling back to requests.
Hugging Face Text-Generation Models & Token Limits
Authentication: ✅ Yes (from .env)
Status: OK = limit found, Gated = license required, NotFound = no config
Fetching text-generation models (sort: downloads)...
Found 200 models. Fetching token limits...


Fetching token limits: 100%|████████████████████████████████████████████████████████| 200/200 [01:52<00:00,  1.78it/s]


Model ID                                           Token Limit     Status    
deepseek-ai/DeepSeek-V4-Flash-0731                 1,048,576       OK        
RadixArk/Kimi-K3-DSpark                            1,048,576       OK        
deepseek-ai/DeepSeek-V4-Flash                      1,048,576       OK        
zai-org/GLM-5.2-FP8                                1,048,576       OK        
zai-org/GLM-5.2                                    1,048,576       OK        
nvidia/GLM-5.2-NVFP4                               1,048,576       OK        
nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 1,048,576       OK        
deepseek-ai/DeepSeek-V4-Pro                        1,048,576       OK        
cyankiwi/MiniCPM-SALA-AWQ-8bit                     524,288         OK        
nvidia/Qwen3.6-35B-A3B-NVFP4                       262,144         OK        
farbodtavakkoli/OTel-2.0-LLM-31B-IT                262,144         OK        
Qwen/Qwen3-4B-Instruct-2507                        262,144     